# Semantic Correspondence — Analysis & paper-grade figures

Standalone notebook: takes the artifacts produced by the training pipeline (locally or downloaded from Colab Drive) and produces **publication-ready tables, plots, and qualitative figures** for the project report.

## Inputs (read from local filesystem)

| Path | Content |
|------|---------|
| `runs/pipeline_exports/pck_results.json` | Aggregate per-run PCK + per-image / per-point lists (always required) |
| `runs/pipeline_exports/pck_results_per_category.json` | Per-category PCK breakdown (recommended) |
| `runs/pipeline_exports/pck_results_by_difficulty_flag.json` | Per-flag (viewpoint / scale / truncation / occlusion) PCK |
| `checkpoints/<backbone>_pretrain.pth` etc. | Pretrained encoder weights (DINOv2/DINOv3/SAM) |
| `checkpoints/<backbone>_lastblocks{1,2,4}_best.pt` | Fine-tuned checkpoints |
| `checkpoints/<backbone>_lora_r8_best.pt` | LoRA checkpoints |
| `checkpoints/*_history.jsonl` | Per-epoch train/val loss |
| `runs/logs/pipeline_*.log` | Training log (used to estimate wall-clock; optional) |
| `data/SPair-71k/` | Required for qualitative inference (Section G) |

## Outputs (`runs/paper_figures/`)

```text
runs/paper_figures/
  tables/    master_pck.{tex,md,csv}, per_category.tex, per_difficulty.tex, efficiency.tex
  figures/   wsa_gain.{pdf,png}, ft_depth.{pdf,png}, lora_vs_ft.{pdf,png},
             per_category_heatmap.{pdf,png}, per_difficulty.{pdf,png}, training_curves.{pdf,png}
  qualitative/  <pair_id>_grid.{pdf,png}, <pair_id>_heatmap.{pdf,png}, index.md
```

## Workflow

**A. Local-only run.** After [`AML_Local.ipynb`](AML_Local.ipynb) finishes, just run this notebook — paths are already correct.

**B. Colab run + local analysis.** After [`AML_Colab.ipynb`](AML_Colab.ipynb) finishes, copy `runs/pipeline_exports/`, `runs/logs/`, and `checkpoints/` from the Drive folder `MyDrive/Colab Notebooks/AML_results/` into your local repo (preserving the layout above). Then run this notebook locally on **CPU or MPS** (no GPU required).

Sections G (qualitative) loads checkpoints and runs short forward passes; the other sections only need the JSON exports.


## A. Setup & inventory

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path.cwd().resolve()
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

PIPELINE_EXPORTS_DIR = REPO_DIR / "runs" / "pipeline_exports"
CHECKPOINTS_DIR      = REPO_DIR / "checkpoints"
SPAIR_ROOT           = REPO_DIR / "data" / "SPair-71k"
LOGS_DIR             = REPO_DIR / "runs" / "logs"
PAPER_OUT_DIR        = REPO_DIR / "runs" / "paper_figures"
TABLES_DIR     = PAPER_OUT_DIR / "tables"
FIGURES_DIR    = PAPER_OUT_DIR / "figures"
QUAL_DIR       = PAPER_OUT_DIR / "qualitative"
for d in (TABLES_DIR, FIGURES_DIR, QUAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Inputs")
for label, p in [
    ("pck_results.json",                     PIPELINE_EXPORTS_DIR / "pck_results.json"),
    ("pck_results_per_category.json",        PIPELINE_EXPORTS_DIR / "pck_results_per_category.json"),
    ("pck_results_by_difficulty_flag.json",  PIPELINE_EXPORTS_DIR / "pck_results_by_difficulty_flag.json"),
    ("checkpoints/",                         CHECKPOINTS_DIR),
    ("data/SPair-71k/",                      SPAIR_ROOT),
    ("runs/logs/",                           LOGS_DIR),
]:
    status = "OK" if (p.is_file() if p.suffix else p.is_dir()) else "MISSING"
    print(f"  [{status}] {label}: {p}")

print("Outputs ->", PAPER_OUT_DIR)


In [ ]:
from evaluation.figures import (
    apply_paper_style, load_pck_exports, build_master_table,
    dataframe_to_latex, dataframe_to_markdown,
    plot_wsa_gain, plot_ft_depth, plot_lora_vs_ft,
    per_category_table, plot_per_category_heatmap,
    per_difficulty_table, plot_per_difficulty_bars,
    plot_wsa_window_sensitivity, plot_dino_layer_sensitivity,
    load_training_histories, plot_training_curves,
    estimate_trainable_params,
)

apply_paper_style()
pck_data = load_pck_exports(PIPELINE_EXPORTS_DIR)
print("Loaded", len(pck_data["pck_results"]), "runs from", PIPELINE_EXPORTS_DIR)
for k, v in pck_data["available"].items():
    print(f"  {k}: {'present' if v else 'missing'}")


## B. Master comparative table

Rows: `(backbone, method, wsa)`. Columns: `pck@α` (image macro) and `pck_pt@α` (point micro) for α ∈ {0.05, 0.1, 0.2}. The LaTeX export bolds the column-wise best so the table can be pasted directly into the paper.


In [ ]:
df_master = build_master_table(pck_data)
print("rows:", len(df_master))
display(df_master.style.format({c: "{:.4f}" for c in df_master.columns
                                if str(c).startswith(("pck@", "pck_pt@"))}).set_caption("Master PCK"))


In [ ]:
csv_path = TABLES_DIR / "master_pck.csv"
df_master.to_csv(csv_path, index=False)
print("Wrote", csv_path)

tex = dataframe_to_latex(df_master, caption="PCK comparison on SPair-71k test split.",
                         label="tab:master_pck")
(TABLES_DIR / "master_pck.tex").write_text(tex, encoding="utf-8")
print("Wrote", TABLES_DIR / "master_pck.tex")

try:
    md = dataframe_to_markdown(df_master)
    (TABLES_DIR / "master_pck.md").write_text(md + "\n", encoding="utf-8")
    print("Wrote", TABLES_DIR / "master_pck.md")
except ImportError:
    print("Markdown export skipped (install `tabulate` to enable).")


## C. Δ-PCK plots

### C.1 WSA gain (sub-pixel refinement, PDF Stage 3)
Pairs `(method, +WSA)` per `(backbone, method)`. Negative bars indicate WSA hurts and would imply a noisy similarity surface.


In [ ]:
for a in (0.05, 0.1, 0.2):
    paths = plot_wsa_gain(df_master, FIGURES_DIR / f"wsa_gain_a{a:g}", alpha=a)
    print("  ->", paths[0].name, "+", paths[1].name)


### C.2 Fine-tune depth (PDF Stage 2 question)
PCK vs unfrozen blocks `N ∈ {1, 2, 4}`, one subplot per backbone. Answers “do more unfrozen blocks help?” and reveals diminishing returns.


In [ ]:
paths = plot_ft_depth(df_master, FIGURES_DIR / "ft_depth")
print("  ->", paths[0].name, "+", paths[1].name)


### C.3 LoRA vs full fine-tune (parameter efficiency)
Trainable parameters (log scale, derived from saved checkpoints) vs PCK — demonstrates the LoRA trade-off.


In [ ]:
paths = plot_lora_vs_ft(df_master, CHECKPOINTS_DIR, FIGURES_DIR / "lora_vs_ft", alpha=0.1)
print("  ->", paths[0].name, "+", paths[1].name)


### C.4 Inference-only ablations (WSA window, DINO layer)

Two sensitivity plots driven by the pipeline sweep exports (`pck_results_wsa_sweep.json`, `pck_results_layer_sweep.json`). Both runs are inference-only on the frozen backbone, so any curvature here reflects the choice of refinement window / extraction depth, not extra training.

In [ ]:
if pck_data["available"].get("wsa_sweep"):
    paths = plot_wsa_window_sensitivity(pck_data, FIGURES_DIR / "wsa_window_sensitivity_a0.1", alpha=0.1)
    print("  ->", paths[0].name, "+", paths[1].name)
else:
    print("  [skip] pck_results_wsa_sweep.json missing")

if pck_data["available"].get("layer_sweep"):
    paths = plot_dino_layer_sensitivity(pck_data, FIGURES_DIR / "dino_layer_sensitivity_a0.1", alpha=0.1)
    print("  ->", paths[0].name, "+", paths[1].name)
else:
    print("  [skip] pck_results_layer_sweep.json missing")

## D. Per-category breakdown

Heatmap (run × SPair category) and the auto-generated LaTeX table for the appendix.


In [ ]:
for a in (0.1, 0.05, 0.2):
    paths = plot_per_category_heatmap(pck_data, FIGURES_DIR / f"per_category_a{a:g}", alpha=a)
    print("  ->", paths[0].name)

pivot_cat = per_category_table(pck_data, alpha=0.1)
if not pivot_cat.empty:
    pivot_cat.round(4).to_csv(TABLES_DIR / "per_category_a0.1.csv")
    print("Wrote", TABLES_DIR / "per_category_a0.1.csv")
    display(pivot_cat.round(4))
else:
    print("No per-category data to plot.")


In [ ]:
import matplotlib.pyplot as plt

if not pivot_cat.empty:
    means = pivot_cat.mean(axis=1).sort_values()
    top5 = means.tail(5)
    bot5 = means.head(5)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].barh(bot5.index, bot5.values, color="tab:red")
    axes[0].set_title("Bottom-5 runs (mean PCK across categories @ 0.1)")
    axes[0].set_xlim(0, 1)
    axes[1].barh(top5.index, top5.values, color="tab:green")
    axes[1].set_title("Top-5 runs")
    axes[1].set_xlim(0, 1)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "per_category_top_bottom.pdf", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "per_category_top_bottom.png", bbox_inches="tight", dpi=150)
    plt.show()
    plt.close(fig)


## E. Per-difficulty (viewpoint / scale / truncation / occlusion)

Each flag is binary in SPair-71k (low vs high). Bigger gaps imply the method is sensitive to that variation factor.


In [ ]:
paths = plot_per_difficulty_bars(pck_data, FIGURES_DIR / "per_difficulty_a0.1", alpha=0.1)
print("  ->", paths[0].name)

df_diff = per_difficulty_table(pck_data, alpha=0.1)
if not df_diff.empty:
    pivot = df_diff.pivot_table(index="run", columns=["flag", "bucket"], values="value").round(4)
    pivot.to_csv(TABLES_DIR / "per_difficulty_a0.1.csv")
    print("Wrote", TABLES_DIR / "per_difficulty_a0.1.csv")
    display(pivot)
else:
    print("No per-difficulty data.")


## F. Training curves

One subplot per training run, with the early-stopping best epoch annotated.


In [ ]:
histories = load_training_histories(CHECKPOINTS_DIR)
print("history files found:", len(histories))
if histories:
    paths = plot_training_curves(CHECKPOINTS_DIR, FIGURES_DIR / "training_curves")
    print("  ->", paths[0].name, "+", paths[1].name)
else:
    print("No *_history.jsonl in", CHECKPOINTS_DIR, "\u2014 skipping.")


## G. Qualitative paper-grade figures

Loads the trained checkpoints, runs short forward passes on **CPU or MPS** (no CUDA required), and produces:

1. **Method-comparison grids** — rows = methods (baseline / FT-LB1/2/4 / LoRA / +WSA variants), columns = source/target with predicted (X) and GT (o) keypoints, color-coded by PCK correctness.
2. **Similarity-heatmap overlays** — cosine similarity surface on the target image for the most informative source keypoints.
3. **Failure-case browser** — the K hardest pairs per run, surfaced from `sd4match_per_image["custom_pck0.1"]["all"]`.
4. **Symmetry-ambiguity diagnostic** — left/right keypoints whose prediction is closer to the symmetric peer than the GT.

**Memory-light by design:** extractors are loaded one at a time per pair (and reused across pairs in the loop). For a comprehensive grid we focus on DINOv2 and DINOv3 (smaller than SAM); SAM can be added by extending `METHOD_SPECS` below.


In [ ]:
import torch

from data.dataset import PreprocessMode, SPair71kPairDataset
from evaluation.qualitative import (
    MethodSpec, find_failure_cases, load_method_extractors,
    render_method_comparison_grid, render_similarity_heatmap_overlay,
)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Qualitative inference device:", DEVICE)

PRETRAINED = {
    "dinov2_vitb14": str(CHECKPOINTS_DIR / "dinov2_vitb14_pretrain.pth"),
    "dinov3_vitb16": str(CHECKPOINTS_DIR / "dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"),
    "sam_vit_b":     str(CHECKPOINTS_DIR / "sam_vit_b_01ec64.pth"),
}

QUAL_HW = (518, 518)
QUAL_BACKBONES = ["dinov2_vitb14", "dinov3_vitb16"]
QUAL_METHODS = ["baseline", "ft_lb2", "lora"]
METHOD_SPECS = [MethodSpec(bb, m) for bb in QUAL_BACKBONES for m in QUAL_METHODS]

missing_pretrained = [k for k, p in PRETRAINED.items()
                      if k in QUAL_BACKBONES and not Path(p).is_file()]
if missing_pretrained:
    print("WARN missing pretrained weights for:", missing_pretrained,
          "\u2014 those backbones will fall back to torch.hub if available.")


In [ ]:
if SPAIR_ROOT.is_dir():
    qual_ds = SPair71kPairDataset(
        spair_root=str(SPAIR_ROOT),
        split="test",
        preprocess=PreprocessMode.FIXED_RESIZE,
        output_size_hw=QUAL_HW,
        normalize=True,
    )
    print("Test pairs:", len(qual_ds))
else:
    qual_ds = None
    print("data/SPair-71k missing; qualitative section will be skipped.")


In [ ]:
extractors = {}
if qual_ds is not None:
    extractors = load_method_extractors(
        METHOD_SPECS, ckpt_dir=CHECKPOINTS_DIR,
        pretrained_paths=PRETRAINED, device=DEVICE,
    )
    print("Loaded", len(extractors), "extractors:")
    for label in extractors:
        print("  -", label)


In [ ]:
import random
import matplotlib.pyplot as plt

NUM_CURATED_PAIRS = 6
RANDOM_SEED = 42

if qual_ds is not None and extractors:
    random.seed(RANDOM_SEED)
    seen_categories = set()
    pair_indices = []
    candidates = list(range(len(qual_ds)))
    random.shuffle(candidates)
    for idx in candidates:
        cat = qual_ds[idx].get("category", "?")
        if cat in seen_categories:
            continue
        seen_categories.add(cat)
        pair_indices.append(idx)
        if len(pair_indices) >= NUM_CURATED_PAIRS:
            break
    print("Curated pairs:", pair_indices)

    qual_index_lines = ["# Qualitative figures index\n"]
    for idx in pair_indices:
        sample = qual_ds[idx]
        pid = str(sample.get("pair_id_str", f"idx{idx}")).replace("/", "_").replace(":", "_")
        fig = render_method_comparison_grid(
            sample, extractors, img_hw=QUAL_HW, alpha=0.1,
        )
        out = QUAL_DIR / f"{pid}_grid"
        fig.savefig(out.with_suffix(".pdf"), bbox_inches="tight")
        fig.savefig(out.with_suffix(".png"), bbox_inches="tight", dpi=150)
        plt.close(fig)
        qual_index_lines.append(f"## {pid} (cat={sample.get('category','?')})\n")
        qual_index_lines.append(f"![grid]({pid}_grid.png)\n")
        print("  wrote", out.name)
    (QUAL_DIR / "index.md").write_text("\n".join(qual_index_lines), encoding="utf-8")
    print("Wrote", QUAL_DIR / "index.md")
else:
    print("Skipping qualitative grids: dataset or extractors unavailable.")


In [ ]:
# Similarity heatmaps for the *first* curated pair, with the DINOv2 baseline (most informative
# for understanding zero-shot saliency). Pick 3 valid keypoints automatically.
if qual_ds is not None and extractors and pair_indices:
    sample = qual_ds[pair_indices[0]]
    n_valid = int(sample["n_valid_keypoints"].view(-1)[0].item())
    kp_idx_list = list(range(min(3, n_valid)))
    target_label = next((lbl for lbl in extractors
                         if lbl.startswith("dinov2_vitb14")), next(iter(extractors)))
    print("Heatmap on:", target_label, "keypoints", kp_idx_list)
    fig = render_similarity_heatmap_overlay(
        sample, extractors[target_label],
        img_hw=QUAL_HW, keypoint_indices=kp_idx_list,
    )
    pid = str(sample.get("pair_id_str", "idx")).replace("/", "_").replace(":", "_")
    out = QUAL_DIR / f"{pid}_heatmap"
    fig.savefig(out.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(out.with_suffix(".png"), bbox_inches="tight", dpi=150)
    plt.close(fig)
    print("Wrote", out.name)


In [ ]:
# Failure-case browser: bottom-3 pairs for each available extractor (using the run name
# embedded in pck_results.json that matches the extractor's MethodSpec).
FAILURE_K = 3

def _spec_to_run_name(spec: MethodSpec) -> str:
    return f"{spec.backbone}_{spec.method}"

if qual_ds is not None and extractors:
    failure_dump = {}
    for spec in METHOD_SPECS:
        run = _spec_to_run_name(spec)
        cases = find_failure_cases(pck_data["pck_results"], run_name=run, alpha=0.1, k=FAILURE_K)
        failure_dump[run] = cases
        print(f"{run}: bottom-{FAILURE_K} = {cases}")
        if not cases:
            continue
        if spec.label not in extractors:
            continue
        for pair_idx, pck_val in cases:
            try:
                sample = qual_ds[pair_idx]
            except IndexError:
                continue
            single = {spec.label: extractors[spec.label]}
            fig = render_method_comparison_grid(sample, single, img_hw=QUAL_HW, alpha=0.1)
            pid = str(sample.get("pair_id_str", f"idx{pair_idx}")).replace("/", "_").replace(":", "_")
            out = QUAL_DIR / f"failure_{run}_{pid}"
            fig.savefig(out.with_suffix(".png"), bbox_inches="tight", dpi=150)
            plt.close(fig)
    import json as _json
    (QUAL_DIR / "failure_cases.json").write_text(
        _json.dumps(failure_dump, indent=2), encoding="utf-8",
    )
    print("Wrote", QUAL_DIR / "failure_cases.json")


## H. Efficiency table

Trainable parameters per `(backbone, method)` and time-to-best-epoch (read from `*_history.jsonl` if available).


In [ ]:
import pandas as pd

rows = []
for bb in ("dinov2_vitb14", "dinov3_vitb16", "sam_vit_b"):
    for n in (1, 2, 4):
        n_params = estimate_trainable_params(bb, f"ft_lb{n}", ckpt_dir=CHECKPOINTS_DIR, last_blocks=n)
        rows.append({"backbone": bb, "method": f"ft_lb{n}", "trainable_params": n_params})
    n_lora = estimate_trainable_params(bb, "lora", ckpt_dir=CHECKPOINTS_DIR, rank=8)
    rows.append({"backbone": bb, "method": "lora", "trainable_params": n_lora})
df_eff = pd.DataFrame(rows)

histories = load_training_histories(CHECKPOINTS_DIR)
best_epochs = {}
for name, recs in histories.items():
    valid = [(r.get("epoch"), r.get("val_loss")) for r in recs if r.get("val_loss") is not None]
    if valid:
        be, _ = min(valid, key=lambda t: t[1])
        best_epochs[name] = int(be)

def _hist_key(row):
    if row["method"].startswith("ft_lb"):
        return f"{row['backbone']}_{row['method']}"
    if row["method"] == "lora":
        return f"{row['backbone']}_lora_r8"
    return None

df_eff["best_epoch"] = [best_epochs.get(_hist_key(r)) for _, r in df_eff.iterrows()]
df_eff.to_csv(TABLES_DIR / "efficiency.csv", index=False)
tex = dataframe_to_latex(
    df_eff, metric_cols=["trainable_params", "best_epoch"], fmt="{:,.0f}",
    caption="Trainable parameters and convergence epoch per (backbone, method).",
    label="tab:efficiency",
)
(TABLES_DIR / "efficiency.tex").write_text(tex, encoding="utf-8")
print("Wrote", TABLES_DIR / "efficiency.csv", "+ .tex")
display(df_eff)


## I. Paper bundle inventory

All files written by this notebook, ready to be `\input{}` / `\includegraphics{}` from a LaTeX paper.


In [ ]:
def _tree(root: Path, prefix: str = "") -> None:
    entries = sorted(root.iterdir(), key=lambda p: (not p.is_dir(), p.name))
    for i, p in enumerate(entries):
        connector = "└── " if i == len(entries) - 1 else "├── "
        if p.is_dir():
            print(prefix + connector + p.name + "/")
            _tree(p, prefix + ("    " if i == len(entries) - 1 else "│   "))
        else:
            size_kb = p.stat().st_size / 1024.0
            print(f"{prefix}{connector}{p.name}  ({size_kb:.1f} KB)")

print(PAPER_OUT_DIR.relative_to(REPO_DIR), "/")
_tree(PAPER_OUT_DIR)


---

**Done.** Tables in `runs/paper_figures/tables/`, figures in `runs/paper_figures/figures/`, qualitative overlays in `runs/paper_figures/qualitative/`. Re-run any individual cell to regenerate a single artifact.